# ISD Lite Weather Data Retrieval and Processing

This notebook uses the `pyisd` library to retrieve weather data from the ISD Lite dataset, processes it using Polars, and saves the results as a Parquet file for later use.

The format of the ISD Lite dataset is described in the [ISD Lite documentation](https://www.ncei.noaa.gov/pub/data/noaa/isd-lite/isd-lite-format.pdf). Each row in the dataset represents a single observation, and the columns include various weather parameters such as temperature, dew point, sea level pressure, wind direction, wind speed, sky condition, and precipitation.

The full ISD dataset does have some extra information we may want to include, such as ceiling and visibility, but for now we will just extract the information from the ISD Lite dataset for simplicity.

In [1]:
# Initialize the ISD Lite client
from pyisd import IsdLite

isd = IsdLite(crs=4326, verbose=True)

/Users/ozzysimpson/Git/Flight-Delay/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
STATION = "724050-13743"  # DCA
START = "2024-01-01"
END = "2025-08-26"  # This is the latest date for which there is ISD data

In [3]:
STATION = "724050-13743"
station_data = isd.get_data(
    start=START, end=END, station_id=STATION, organize_by="location"
)
weather_data = station_data[STATION]
weather_data.head(20)

100%|██████████| 1/1 [00:00<00:00,  1.57it/s]


,temp,dewtemp,pressure,winddirection,windspeed,skycoverage,precipitation-1h,precipitation-6h
2024-01-01 00:00:00,7.2,-0.6,1016.3,160.0,3.1,6.0,NaN,NaN
2024-01-01 01:00:00,6.7,0.0,1016.4,190.0,3.1,NaN,0.0,NaN
2024-01-01 02:00:00,6.7,0.0,1016.6,190.0,3.6,NaN,0.0,NaN
2024-01-01 03:00:00,6.7,0.6,1016.6,170.0,2.1,8.0,0.0,NaN
2024-01-01 04:00:00,6.1,0.0,1016.1,130.0,2.6,NaN,0.0,NaN
2024-01-01 05:00:00,5.6,0.0,1015.7,0.0,0.0,2.0,0.0,NaN
2024-01-01 06:00:00,5.6,0.6,1015.4,120.0,1.5,2.0,0.0,NaN
2024-01-01 07:00:00,5.0,0.6,1015.3,70.0,1.5,NaN,0.0,NaN
2024-01-01 08:00:00,5.0,1.1,1015.3,0.0,0.0,NaN,-1.0,NaN
2024-01-01 09:00:00,5.6,2.2,1015.3,0.0,0.0,8.0,-1.0,NaN


In [4]:
# Convert the data to Polars DataFrame, with the index as a column named 'timestamp'
import polars as pl

weather_df = pl.DataFrame(weather_data.reset_index())
weather_df = weather_df.rename({"index": "timestamp"})
weather_df.head()

timestamp,temp,dewtemp,pressure,winddirection,windspeed,skycoverage,precipitation-1h,precipitation-6h
datetime[μs],f64,f64,f64,f64,f64,f64,f64,f64
2024-01-01 00:00:00,7.2,-0.6,1016.3,160.0,3.1,6.0,null,null
2024-01-01 01:00:00,6.7,0.0,1016.4,190.0,3.1,null,0.0,null
2024-01-01 02:00:00,6.7,0.0,1016.6,190.0,3.6,null,0.0,null
2024-01-01 03:00:00,6.7,0.6,1016.6,170.0,2.1,8.0,0.0,null
2024-01-01 04:00:00,6.1,0.0,1016.1,130.0,2.6,null,0.0,null


There are a few special cases to handle in the data:
- For precipitation, a value of -1 indicates a trace amount (less than 0.01 inches). We will create a new column to indicate whether the precipitation was a trace amount and replace the original -1 value with 0 for the precipitation amount.
- The sky coverage is represented by a code (e.g., 0 for clear, 1 for one okta, etc.). We'll convert the numeric codes into the actual labels for better readability.

In [5]:
# Replace trace precipitation (-1) with 0 and add a flag
weather_df = weather_df.with_columns(
    [
        # precipitation-1h_trace: 1 if original value was -1 (trace), 0 otherwise
        pl.when(pl.col("precipitation-1h") == -1.0)
        .then(1)
        .otherwise(0)
        .alias("precip-1h_trace"),
        # Replace -1 with 0 for the precipitation amount
        pl.when(pl.col("precipitation-1h") == -1.0)
        .then(0)
        .otherwise(pl.col("precipitation-1h"))
        .alias("precipitation-1h"),
        # precipitation-6h_trace: 1 if original value was -1 (trace), 0 otherwise
        pl.when(pl.col("precipitation-6h") == -1.0)
        .then(1)
        .otherwise(0)
        .alias("precip-6h_trace"),
        # Replace -1 with 0 for the precipitation amount
        pl.when(pl.col("precipitation-6h") == -1.0)
        .then(0)
        .otherwise(pl.col("precipitation-6h"))
        .alias("precipitation-6h"),
    ]
)

# Map numeric sky coverage codes to categorical labels
sky_coverage_map = {
    0: "clear",
    1: "1_okta",
    2: "2-3_oktas",
    3: "4_oktas",
    4: "5_oktas",
    5: "6_oktas",
    6: "7-8_oktas",
    7: "9+_oktas_not_full",
    8: "10_oktas_overcast",
    9: "obscured",
    10: "partial_obscured",
    11: "thin_scattered",
    12: "scattered",
    13: "dark_scattered",
    14: "thin_broken",
    15: "broken",
    16: "dark_broken",
    17: "thin_overcast",
    18: "overcast_standard",
    19: "dark_overcast",
    -9999: "missing",
}

# Create sky coverage label column
weather_df = weather_df.with_columns(
    [
        pl.col("skycoverage")
        .map_elements(
            lambda x: sky_coverage_map.get(x, "unknown"), return_dtype=pl.Utf8
        )
        .alias("skycoverage")
    ]
)

weather_df.head(100)

timestamp,temp,dewtemp,pressure,winddirection,windspeed,skycoverage,precipitation-1h,precipitation-6h,precip-1h_trace,precip-6h_trace
datetime[μs],f64,f64,f64,f64,f64,str,f64,f64,i32,i32
2024-01-01 00:00:00,7.2,-0.6,1016.3,160.0,3.1,"""7-8_oktas""",null,null,0,0
2024-01-01 01:00:00,6.7,0.0,1016.4,190.0,3.1,null,0.0,null,0,0
2024-01-01 02:00:00,6.7,0.0,1016.6,190.0,3.6,null,0.0,null,0,0
2024-01-01 03:00:00,6.7,0.6,1016.6,170.0,2.1,"""10_oktas_overcast""",0.0,null,0,0
2024-01-01 04:00:00,6.1,0.0,1016.1,130.0,2.6,null,0.0,null,0,0
…,…,…,…,…,…,…,…,…,…,…
2024-01-04 23:00:00,3.9,-7.2,1022.7,320.0,5.1,"""2-3_oktas""",0.0,null,0,0
2024-01-05 00:00:00,2.8,-7.2,1023.4,340.0,6.2,"""clear""",0.0,null,0,0
2024-01-05 01:00:00,2.2,-7.2,1024.0,320.0,5.1,"""clear""",0.0,null,0,0


# ISD Full Weather Data Retrieval and Processing

While the ISD Lite dataset gives us most of the weather data we want in a nice format, it doesn't include visibility and ceiling height, which may be important features. To get those, we need to use the full ISD dataset, which is more complex to work with.

In [6]:
INPUT = [
    "72405013743-2024.csv",
    "72405013743-2025.csv",
]  # List of input CSV files containing ISD weather data
OUTPUT = "isd_weather_data.parquet"  # Output Parquet file to store the processed ISD weather data

COLUMNS_WANTED = [
    "DATE",
    "VIS",  # visibility
    "CIG",  # ceiling height
]

# We'll later filter out rows where report_type is not in this list, as these are the types of reports we want to keep
REPORT_TYPES = [
    "AUTO",  # Report from an automatic station
    "FM-12",  # SYNOP Report of surface observation from a fixed land station
    "FM-15",  # METAR Aviation routine weather report
    "FM-16",  # SPECI Aviation selected special weather report
    "S-S-A",  # Synoptic, airways, and auto merged report
    "SA-AU",  # Airways and auto merged report
    "SY-AU",  # Synoptic and auto merged report
    "SY-MT",  # Synoptic and METAR merged report
]

In [7]:
def parse_isd_file(file_path: str) -> pl.DataFrame:
    """
    Parses an ISD CSV file and returns a DataFrame with the desired columns and filtered by report type.

    Parameters:
        file_path (str): The path to the ISD CSV file.
    Returns:
        pl.DataFrame: A DataFrame containing the parsed and filtered ISD weather data.
    """
    # Read the CSV file into a DataFrame
    df = pl.read_csv(
        file_path, columns=COLUMNS_WANTED + ["REPORT_TYPE"]
    )  # we don't want to actually include the report type in the final output, but we need it for filtering, so we read it in here and will drop it later

    # Filter the DataFrame to include only rows with the desired report types
    df = df.filter(pl.col("REPORT_TYPE").is_in(REPORT_TYPES))

    # Drop the 'report_type' column as it's no longer needed
    df = df.drop("REPORT_TYPE")

    # Convert the 'DATE' column to a datetime type
    df = df.with_columns(
        pl.col("DATE").str.to_datetime(format="%Y-%m-%dT%H:%M:%S")
    ).rename({"DATE": "timestamp"})

    # ====================================
    # VIS cleanup
    ## VIS format: "DISTANCE,DISTANCE_Q,VARIABILITY,VARIABILITY_Q" (e.g. "016000,1,9,9")
    df = df.with_columns(
        [pl.col("VIS").str.split(",").alias("vis_parts")]
    ).with_columns(
        [
            pl.col("vis_parts").list.get(0).cast(pl.Int32).alias("visibility_m"),
            pl.col("vis_parts").list.get(1).alias("visibility_q"),
            pl.col("vis_parts").list.get(2).alias("visibility_variability"),
            pl.col("vis_parts").list.get(3).alias("visibility_variability_q"),
        ]
    )

    ## Drop missing values, if other columns are "bad", make all visibility columns null for that row
    df = df.with_columns(
        [
            (
                (pl.col("visibility_m") != 999999)
                & pl.col("visibility_q").is_in(["0", "1", "4", "5", "9"])
                & pl.col("visibility_variability_q").is_in(["0", "1", "4", "5", "9"])
            ).alias("visibility_valid")
        ]
    )

    ## If visibility is invalid, set visibility columns to null
    df = df.with_columns(
        [
            pl.when(pl.col("visibility_valid"))
            .then(pl.col("visibility_m"))
            .otherwise(None)
            .alias("visibility_m"),
            pl.when(pl.col("visibility_valid"))
            .then(pl.col("visibility_q"))
            .otherwise(None)
            .alias("visibility_q"),
        ]
    )

    ## Drop intermediate columns we don't need anymore
    df = df.drop(
        [
            "VIS",
            "vis_parts",
            "visibility_q",
            "visibility_variability_q",
            "visibility_valid",
            "visibility_variability",
        ]
    )

    # ====================================
    # CIG cleanup
    ## CIG format: "CEILING_HEIGHT_DIM,CEILING_Q,CEILING_DETERMINATION_CODE,CAVOK_FLAG" (e.g. "9999,1,9,N")
    df = df.with_columns(
        [pl.col("CIG").str.split(",").alias("cig_parts")]
    ).with_columns(
        [
            pl.col("cig_parts").list.get(0).cast(pl.Int32).alias("ceiling_height_m"),
            pl.col("cig_parts").list.get(1).alias("ceiling_q"),
            pl.col("cig_parts").list.get(2).alias("ceiling_determination_code"),
            pl.col("cig_parts").list.get(3).alias("cavok_flag"),
        ]
    )

    ## Drop missing values, if other columns are "bad", make all ceiling columns null for that row
    df = df.with_columns(
        [
            (
                (pl.col("ceiling_height_m") != 99999)
                & (pl.col("ceiling_q").is_in(["0", "1", "4", "5", "9"]))
                & (pl.col("ceiling_determination_code") != "9")
            ).alias("ceiling_valid")
        ]
    )

    ## If ceiling is invalid, set ceiling columns to null
    df = df.with_columns(
        [
            pl.when(pl.col("ceiling_valid"))
            .then(pl.col("ceiling_height_m"))
            .otherwise(None)
            .alias("ceiling_height_m"),
        ]
    )

    ## Drop intermediate columns we don't need anymore
    df = df.drop(
        [
            "CIG",
            "cig_parts",
            "ceiling_valid",
            "cavok_flag",
            "ceiling_q",
            "ceiling_determination_code",
        ]
    )

    return df

In [8]:
dfs = []  # List to hold DataFrames for each file

for file in INPUT:
    df = parse_isd_file(file)
    dfs.append(df)

# Concatenate all the DataFrames into a single DataFrame
isd_full_df = pl.concat(dfs)
isd_full_df.head()

timestamp,visibility_m,ceiling_height_m
datetime[μs],i32,i32
2024-01-01 00:00:00,16000,null
2024-01-01 00:52:00,16093,3353
2024-01-01 01:52:00,16093,1676
2024-01-01 02:52:00,16093,1524
2024-01-01 03:00:00,16000,null


# Merge ISD Lite and ISD Full Datasets

In [9]:
# Merge the ISD full dataset into the ISD Lite dataset using an asof join on the timestamp, so we get the visibility and ceiling information for each timestamp in the ISD Lite dataset. We use a backward join strategy, so we get the most recent ISD full observation that is at or before the ISD Lite timestamp.
final_df = weather_df.join_asof(isd_full_df, on="timestamp", strategy="backward")
final_df.head()

timestamp,temp,dewtemp,pressure,winddirection,windspeed,skycoverage,precipitation-1h,precipitation-6h,precip-1h_trace,precip-6h_trace,visibility_m,ceiling_height_m
datetime[μs],f64,f64,f64,f64,f64,str,f64,f64,i32,i32,i32,i32
2024-01-01 00:00:00,7.2,-0.6,1016.3,160.0,3.1,"""7-8_oktas""",null,null,0,0,16000,null
2024-01-01 01:00:00,6.7,0.0,1016.4,190.0,3.1,null,0.0,null,0,0,16093,3353
2024-01-01 02:00:00,6.7,0.0,1016.6,190.0,3.6,null,0.0,null,0,0,16093,1676
2024-01-01 03:00:00,6.7,0.6,1016.6,170.0,2.1,"""10_oktas_overcast""",0.0,null,0,0,16000,null
2024-01-01 04:00:00,6.1,0.0,1016.1,130.0,2.6,null,0.0,null,0,0,16093,2134


In [10]:
# Save as parquet file
final_df.write_parquet("isd_weather_data.parquet")